# A graph you were handed

Someone gives you a causal diagram. It came from a domain expert, or from a
paper, or from the last person in the role. Every estimate you produce will be
conditional on it — `identify_effect` says so out loud, returning an assumption
named `graph_is_correct` in state **unverified**.

But a graph is not merely an assumption. It is a *falsifiable* one: it implies
conditional independencies, and the data either shows them or it does not. This
notebook takes a handed-down graph, asks it what it claims, and checks.

In [ ]:
import numpy as np
import plotly.graph_objects as go

from axiom.display import enable

enable();  # every axiom result renders itself from here on

# One palette, stated once. Light surface, thin marks, recessive chrome.
SURFACE, INK, SECOND, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
GOOD, CRITICAL = "#0ca30c", "#d03b3b"          # status: reserved, never a series colour
NODE_FILL, NODE_RING = "#ffffff", "#c3c2b7"
RADIUS = 0.13


def _shorten(x0, y0, x1, y1, back=RADIUS):
    """Pull an edge back to the rim of each node circle so arrowheads land cleanly."""
    dx, dy = x1 - x0, y1 - y0
    length = np.hypot(dx, dy) or 1.0
    ux, uy = dx / length, dy / length
    return x0 + ux * back, y0 + uy * back, x1 - ux * back, y1 - uy * back


def graph_figure(positions, edges, title, height=330, subtitle=""):
    """A node-link diagram. `edges` are (source, target, colour, dash)."""
    fig = go.Figure()
    for source, target, colour, dash in edges:
        x0, y0, x1, y1 = _shorten(*positions[source], *positions[target])
        fig.add_annotation(
            x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=1.1, arrowwidth=2, arrowcolor=colour,
            standoff=0, opacity=1.0,
        )
        if dash == "dash":  # a dashed overlay marks an edge under discussion
            fig.add_shape(type="line", x0=x0, y0=y0, x1=x1, y1=y1,
                          line=dict(color=colour, width=2, dash="dot"))
    names = list(positions)
    fig.add_trace(go.Scatter(
        x=[positions[n][0] for n in names], y=[positions[n][1] for n in names],
        mode="markers+text", text=names, textposition="middle center",
        textfont=dict(size=12, color=INK),
        marker=dict(size=44, color=NODE_FILL, line=dict(color=NODE_RING, width=1.5)),
        hoverinfo="text", hovertext=names, showlegend=False,
    ))
    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color=INK), x=0.01,
                   subtitle=dict(text=subtitle, font=dict(size=12, color=SECOND))),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, height=height,
        margin=dict(l=10, r=10, t=64 if subtitle else 48, b=10),
        xaxis=dict(visible=False, range=[-0.25, 2.25]),
        yaxis=dict(visible=False, range=[-0.35, 1.6]),
    )
    return fig

## The greenhouse

Five measurements from a growing trial. The expert's diagram says: light warms
the air and drives growth, irrigation drives growth, and growth is what
determines the final yield.

In [ ]:
from axiom.identify import CausalGraph

proposed = CausalGraph.from_edges(
    "light -> heat, light -> growth, water -> growth, growth -> yield",
    name="the expert's diagram",
)

POSITIONS = {
    "light": (0.0, 1.0), "heat": (1.0, 1.25), "water": (0.0, 0.0),
    "growth": (1.15, 0.5), "yield": (2.1, 0.5),
}
graph_figure(
    POSITIONS,
    [(a, b, BLUE, "solid") for a, b in proposed.edges],
    "The diagram you were handed",
    subtitle="Four arrows, and — just as importantly — six pairs with no arrow between them",
).show()

## What the diagram claims

The arrows are the part people discuss. The **missing** arrows are the part that
can be tested: each one is a claim that two variables are independent once the
right things are held fixed.

`implied_independencies` reads them off. Each claim corresponds to exactly one
absent edge — which is what will let a failure point at a repair.

In [ ]:
from axiom.diagnose import ImpliedIndependence, implied_independencies

claims = implied_independencies(proposed)
print(f"{len(claims)} claims, one per pair with no edge between them:\n")
for implication, pair in claims:
    assert isinstance(implication, ImpliedIndependence)
    print(f"  no edge {pair[0]:>6} - {pair[1]:<7}  =>  {implication.describe()}")

## The data

Now the trial's measurements. (Simulated here, so the notebook can show what a
refutation looks like when it lands — but nothing below looks at how they were
made.)

In [ ]:
from axiom.discover import Dataset

rng = np.random.default_rng(11)
n = 2000
light = rng.normal(size=n)
water = rng.normal(size=n)
heat = 1.3 * light + rng.normal(size=n)
growth = 0.9 * light + 0.8 * water + 1.1 * heat + rng.normal(size=n)
yield_ = 1.4 * growth + rng.normal(size=n)

trial = Dataset.observational(
    np.column_stack([light, heat, water, growth, yield_]),
    ["light", "heat", "water", "growth", "yield"],
)
print(trial.n_rows, "rows,", len(trial.names), "variables:", trial.names)

## One test at a time

`PartialCorrelation` is the primitive underneath: it returns a p-value **and**
an effect size, and the reason for both is a sample-size problem that runs in
two directions. With few rows nothing is ever rejected and a graph passes by
being untested; with a million rows everything is rejected, because every graph
is an idealization and idealizations are detectably false at scale.

In [ ]:
from axiom.discover import IndependenceResult, PartialCorrelation

tester = PartialCorrelation(trial)
for x, y, given in [("heat", "growth", ["light"]), ("water", "heat", ["light"])]:
    result = tester.test(x, y, given)
    assert isinstance(result, IndependenceResult)
    print(" ", result.describe())

## All of them at once

`refute_structure` tests every claim, corrects for having asked many questions,
and decides with **both** thresholds: an implication is refuted when its
adjusted p-value is below `alpha` *and* its partial correlation is at least
`effect_threshold`.

In [ ]:
from axiom.diagnose import Correction, IndependenceCheck, StructureRefutation, refute_structure

correction: Correction = "holm"
report = refute_structure(proposed, trial, alpha=0.05, correction=correction)
assert isinstance(report, StructureRefutation)

print("verdict:", report.verdict.status)
print(report.summary())
print()
for check in sorted(report.checks, key=lambda c: -c.effect):
    assert isinstance(check, IndependenceCheck)
    print(" ", check.describe())

In [ ]:
checks = sorted(report.checks, key=lambda c: -c.effect)
labels = [f"{c.implication.x}–{c.implication.y}" for c in checks]
effects = [c.effect for c in checks]
evidence = [-np.log10(max(c.adjusted_p, 1e-300)) for c in checks]
refuted = [c.refuted for c in checks]

fig = go.Figure()
for flag, colour, symbol, label in [
    (True, CRITICAL, "x", "refuted"),
    (False, GOOD, "circle", "held"),
]:
    keep = [i for i, r in enumerate(refuted) if r == flag]
    if not keep:
        continue
    # Label the refuted points only: the held ones cluster in one corner and
    # five labels there would be a smudge. Hover and the table above carry them.
    fig.add_trace(go.Scatter(
        x=[effects[i] for i in keep], y=[min(evidence[i], 60) for i in keep],
        mode="markers+text" if flag else "markers", name=label,
        text=[labels[i] for i in keep] if flag else None, textposition="top center",
        textfont=dict(size=11, color=SECOND),
        marker=dict(size=13, color=colour, symbol=symbol,
                    line=dict(color=SURFACE, width=2)),
        hovertemplate="%{text}<br>|r| = %{x:.3f}<br>evidence = %{y:.1f}<extra></extra>",
    ))
fig.add_hline(y=-np.log10(0.05), line=dict(color=MUTED, width=1, dash="dot"),
              annotation_text="alpha = 0.05", annotation_font=dict(size=11, color=MUTED))
fig.update_layout(
    title=dict(text="Every claim the diagram makes, tested",
               font=dict(size=15, color=INK), x=0.01,
               subtitle=dict(text="One point per absent edge. Far right = a big dependence "
                                  "the diagram says should not be there.",
                             font=dict(size=12, color=SECOND))),
    xaxis=dict(title="effect size |partial correlation|", gridcolor=GRID, zeroline=False,
               range=[-0.02, max(effects) * 1.25 + 0.02], color=SECOND),
    yaxis=dict(title="evidence  −log₁₀(adjusted p)", gridcolor=GRID, zeroline=False, color=SECOND),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, height=400,
    margin=dict(l=60, r=20, t=76, b=52),
    legend=dict(orientation="h", y=1.0, x=1.0, xanchor="right", yanchor="bottom",
                font=dict(size=11, color=SECOND)),
)
fig.show()

One claim is far to the right and far up: **heat and growth are strongly
dependent given light**, and the diagram says they should not be. Everything
else sits at the bottom-left, where a claim that holds belongs.

The point of ranking by effect size rather than by p-value alone: the refuted
claim is not merely *significant*, it is large. That is what makes it worth
acting on.

## A refutation that names its repair

Because each claim is the absence of one edge, a failure points at exactly the
edge whose absence the data denies. `implicated` lists them worst first — not
"your graph is wrong" but "this edge is missing".

In [ ]:
print("verdict :", report.verdict.status)
print("reason  :", report.verdict.reason)
print()
print("assumption state:", [(a.name, a.state) for a in report.verdict.assumptions])
print("edges implicated:", [(a, b, round(r, 3)) for a, b, r in report.implicated])

In [ ]:
repaired = CausalGraph.from_edges(
    "light -> heat, light -> growth, water -> growth, growth -> yield, heat -> growth",
    name="repaired",
)
added = {("heat", "growth")}
graph_figure(
    POSITIONS,
    [(a, b, ORANGE if (a, b) in added else BLUE, "dash" if (a, b) in added else "solid")
     for a, b in repaired.edges],
    "The repair the data asked for",
    subtitle="The orange arrow is the one the refutation named — heat drives growth directly",
).show()

In [ ]:
after = refute_structure(repaired, trial, alpha=0.05)
print("verdict:", after.verdict.status)
print(after.summary())
print()
print(after.verdict.reason)

## Surviving is not passing

The repaired graph is **downgraded**, not identified, and `graph_is_correct`
stays **unverified**. That is deliberate. A test that fails to reject is not
evidence of independence: it may be underpowered, or the dependence may be
nonlinear and invisible to a correlation.

So the honest summary is *"this graph made five checkable claims and the data
contradicted none of them"* — which is worth a great deal more than an
unexamined diagram, and is still not proof.

## The threshold is the question

At `n = 2000` a partial correlation of 0.05 is invisible. At `n = 200000` it is
overwhelming — and still meaningless. Watch a *deliberately tiny* extra
dependence move from "undetectable" to "significant" without ever becoming
important, and watch `effect_threshold` be the thing that keeps the answer
sensible.

In [ ]:
from plotly.subplots import make_subplots


def trial_with_leak(rows, leak, seed=5):
    r = np.random.default_rng(seed)
    li = r.normal(size=rows)
    wa = r.normal(size=rows)
    he = 1.3 * li + r.normal(size=rows)
    gr = 0.9 * li + 0.8 * wa + 1.1 * he + r.normal(size=rows)
    yi = 1.4 * gr + leak * wa + r.normal(size=rows)      # a whisper of a path
    return Dataset.observational(
        np.column_stack([li, he, wa, gr, yi]),
        ["light", "heat", "water", "growth", "yield"],
    )


# One draw, read at growing prefixes -- so what changes is the sample size and
# nothing else.
big = trial_with_leak(200_000, leak=0.05)
sizes = [500, 2_000, 10_000, 50_000, 200_000]
effects, evidence = [], []
for rows in sizes:
    subset = Dataset.observational(big.values[:rows], list(big.names))
    at_n = refute_structure(repaired, subset, effect_threshold=0.0)
    leak_check = next(
        c for c in at_n.checks
        if {c.implication.x, c.implication.y} == {"water", "yield"}
    )
    effects.append(leak_check.effect)
    evidence.append(-np.log10(max(leak_check.adjusted_p, 1e-300)))

print("the claim under test:", leak_check.implication.describe())
for rows, e, v in zip(sizes, effects, evidence):
    print(f"  n = {rows:>7}   |r| = {e:.4f}   evidence = {v:5.1f}")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "the dependence itself", "the evidence for it"))

fig.add_trace(go.Scatter(x=sizes, y=effects, mode="lines+markers",
                         line=dict(color=BLUE, width=2), marker=dict(size=9),
                         hovertemplate="n = %{x}<br>|r| = %{y:.4f}<extra></extra>"),
              row=1, col=1)
fig.add_hline(y=0.1, line=dict(color=MUTED, width=1, dash="dot"), row=1, col=1,
              annotation_text="a threshold worth acting on",
              annotation_font=dict(size=10, color=MUTED))

fig.add_trace(go.Scatter(x=sizes, y=evidence, mode="lines+markers",
                         line=dict(color=CRITICAL, width=2), marker=dict(size=9),
                         hovertemplate="n = %{x}<br>−log₁₀p = %{y:.1f}<extra></extra>"),
              row=1, col=2)
fig.add_hline(y=-np.log10(0.05), line=dict(color=MUTED, width=1, dash="dot"), row=1, col=2,
              annotation_text="alpha = 0.05", annotation_font=dict(size=10, color=MUTED))

fig.update_xaxes(title="rows", type="log", gridcolor=GRID, color=SECOND,
                 tickvals=sizes, ticktext=["500", "2k", "10k", "50k", "200k"])
fig.update_yaxes(gridcolor=GRID, color=SECOND, rangemode="tozero")
fig.update_yaxes(title="|partial correlation|", row=1, col=1)
fig.update_yaxes(title="−log₁₀(adjusted p)", row=1, col=2)
fig.update_layout(
    title=dict(text="The same 0.05 dependence, read at five sample sizes",
               font=dict(size=15, color=INK), x=0.01,
               subtitle=dict(text="Left: what is there — it settles. "
                                  "Right: how sure we are — it never stops.",
                             font=dict(size=12, color=SECOND))),
    showlegend=False, paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, height=360,
    margin=dict(l=60, r=20, t=90, b=52),
)
fig.for_each_annotation(lambda a: a.update(font=dict(size=12, color=SECOND))
                        if a.text in ("the dependence itself", "the evidence for it") else None)
fig.show()

The two panels behave completely differently, and that is the whole point.

The left one **settles**. At 500 rows the estimate reads 0.13 — small samples
inflate a correlation's magnitude — but it converges to the 0.05 that is
actually there and stays well under the line by 10,000 rows. Past that, more
data does not change the answer, because the answer was never in doubt: the
dependence is small.

The right one **never stops**. Evidence against "exactly zero" accumulates
without limit, because the dependence is not exactly zero and enough rows can
prove that about anything.

So a rule reading only the right panel refutes this graph the moment the study
gets big enough. A rule reading both stops as soon as the left panel settles
below what would change a decision — which is the behaviour you want from a
study that keeps growing.

This is the same lesson as the rank tolerance in `design.identifiability` and
the sparsity penalty in `discover` — **the threshold is the question, so it is
a parameter and never a constant.**

## Where this sits

    draw  ->  **refute the structure**  ->  repair  ->  identify  ->
    estimate  ->  refute the estimate  ->  decide

`diagnose.refute` already covered the second refutation — placebo treatments,
permutations, subsets, added noise, all of which perturb the data and refit.
This is the first one, and it is cheaper: no refit, no estimator, just the
claims the diagram already made.

The next notebook picks up where the repair left off. A graph that survives is
still one graph; `nbs/discover/02` asks what the data would have said if nobody
had handed you a diagram at all — and what to do about the parts it cannot
settle.